### Import Libraries 

In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta, timezone
import ast
from pathlib import Path
import nbformat

from pathlib import Path
import shutil


pd.set_option("display.max_columns", None)

### Connect to MooMoo API

In [4]:
import requests

url = "https://webapi.moomoo.com/oauth2/register"

payload = {
    "redirect_uris": ["http://localhost:60355/callback"],
    "token_endpoint_auth_method": "none",
    "grant_types": ["authorization_code", "refresh_token"],
    "response_types": ["code"],
    "client_name": "Moomoo Margin Risk Assistant"
}

response = requests.post(url, json=payload, timeout=20)

print("Status:", response.status_code)

if response.ok:
    registration = response.json()
    client_id = registration["client_id"]
    print("Registration successful!")
    print("Client ID:", client_id)
else:
    print("Error:", response.text[:500])

Status: 201
Registration successful!
Client ID: 47d019d7-807a-4350-900b-3535699bef95


In [5]:
import secrets
import hashlib
import base64
from urllib.parse import urlencode

code_verifier = secrets.token_urlsafe(64)
state = secrets.token_urlsafe(24)

code_challenge = base64.urlsafe_b64encode(
    hashlib.sha256(code_verifier.encode()).digest()
).decode().rstrip("=")

params = {
    "client_id": client_id,
    "response_type": "code",
    "redirect_uri": "http://localhost:60355/callback",
    "scope": "quote:read",
    "state": state,
    "code_challenge": code_challenge,
    "code_challenge_method": "S256"
}

auth_url = (
    "https://webapi.moomoo.com/oauth2/authorize/confirm?"
    + urlencode(params)
)

print(auth_url)

https://webapi.moomoo.com/oauth2/authorize/confirm?client_id=47d019d7-807a-4350-900b-3535699bef95&response_type=code&redirect_uri=http%3A%2F%2Flocalhost%3A60355%2Fcallback&scope=quote%3Aread&state=TZmyRXz6c9fxzpZn7mbJgGImO_lOELk-&code_challenge=qQLmmhz_6LMcyS_zftX04q68LNPhSER9Oz0ggM8Gl-I&code_challenge_method=S256


In [6]:
from urllib.parse import urlparse, parse_qs
from getpass import getpass

# Default to no authentication
access_token = None

redirected_url = getpass(
    "Paste redirected URL (press Enter to skip): "
).strip()

if redirected_url:

    query = parse_qs(urlparse(redirected_url).query)

    returned_state = query.get("state", [None])[0]
    auth_code = query.get("code", [None])[0]

    if returned_state is None or auth_code is None:
        print("Incomplete OAuth URL. Using fallback.")

    elif returned_state != state:
        print("OAuth state mismatch. Using fallback.")

    else:
        print("Authorization code received!")

        try:
            token_response = requests.post(
                "https://webapi.moomoo.com/oauth2/token",
                data={
                    "grant_type": "authorization_code",
                    "code": auth_code,
                    "client_id": client_id,
                    "redirect_uri": "http://localhost:60355/callback",
                    "code_verifier": code_verifier
                },
                timeout=20
            )

            token_response.raise_for_status()
            access_token = token_response.json()["access_token"]

            print("moomoo authentication successful!")

        except (requests.RequestException, KeyError, ValueError):
            print("moomoo authentication unavailable. Using fallback.")

Paste redirected URL (press Enter to skip):  ········


Authorization code received!
moomoo authentication successful!


In [7]:
import requests
import pandas as pd
import numpy as np
import yfinance as yf

tickers = ["NVDA", "TSLA", "AAPL", "MSFT"]

# Read your existing CSV as a frozen backup
backup_prices = pd.read_csv("../../data/stock_prices.csv")

def fetch_stock_prices(tickers):

    results = {}

    # 1. TRY MOOMOO API
    try:
        response = requests.post(
            "https://webapi.moomoo.com/api/v1.0/quote/snapshot",
            headers={
                "Authorization": f"Bearer {access_token}",
                "Content-Type": "application/json"
            },
            json={
                "code_list": [f"US.{t}" for t in tickers]
            },
            timeout=15
        )

        response.raise_for_status()
        data = response.json()

        if data.get("ret_code") != 0:
            raise ValueError(data.get("ret_msg"))

        for item in data.get("data", {}).get("snapshot_list", []):
            ticker = item["code"].replace("US.", "")
            price = float(item["last_price"])

            if ticker in tickers and np.isfinite(price) and price > 0:
                results[ticker] = {
                    "ticker": ticker,
                    "price": price,
                    "timestamp": str(item.get("data_date", "")),
                    "source": "moomoo"
                }

    except Exception as e:
        print("moomoo unavailable:", e)

    # 2. FALLBACK TO YFINANCE
    for ticker in tickers:

        if ticker in results:
            continue

        try:
            history = yf.Ticker(ticker).history(period="5d")

            if history.empty:
                raise ValueError("No Yahoo data")

            price = float(history["Close"].iloc[-1])

            if not np.isfinite(price) or price <= 0:
                raise ValueError("Invalid price")

            results[ticker] = {
                "ticker": ticker,
                "price": price,
                "timestamp": str(history.index[-1].date()),
                "source": "yfinance"
            }

        except Exception as e:
            print(f"{ticker}: yfinance unavailable:", e)

        # 3. FINAL FALLBACK: CSV
        if ticker not in results:

            row = backup_prices[
                backup_prices["ticker"] == ticker
            ]

            if row.empty:
                raise ValueError(
                    f"No available price for {ticker}"
                )

            row = row.iloc[0]

            results[ticker] = {
                "ticker": ticker,
                "price": float(row["price"]),
                "timestamp": str(row["timestamp"]),
                "source": "CSV fallback"
            }

    return pd.DataFrame(results.values())

In [8]:
stock_prices = fetch_stock_prices(tickers)

display(stock_prices)

,ticker,price,timestamp,source
0,NVDA,206.870,2026-11-02,moomoo
1,TSLA,410.880,2026-11-02,moomoo
2,AAPL,339.365,2026-11-02,moomoo
3,MSFT,523.845,2026-11-02,moomoo


### 1. Convert MooMoo Data into Format

In [10]:
import pandas as pd
import numpy as np
import requests
import yfinance as yf

tickers = ["NVDA", "TSLA", "AAPL", "MSFT"]

# CSV is the final backup
backup_prices = pd.read_csv("../../data/stock_prices.csv")

def fetch_stock_prices(tickers):

    results = {}

    # 1. Try moomoo first
    if globals().get("access_token"):

        try:
            response = requests.post(
                "https://webapi.moomoo.com/api/v1.0/quote/snapshot",
                headers={
                    "Authorization": f"Bearer {access_token}",
                    "Content-Type": "application/json"
                },
                json={
                    "code_list": [f"US.{t}" for t in tickers]
                },
                timeout=15
            )

            response.raise_for_status()
            data = response.json()

            if data.get("ret_code") != 0:
                raise ValueError(data.get("ret_msg"))

            for item in data.get("data", {}).get("snapshot_list", []):

                ticker = item["code"].replace("US.", "")
                price = float(item["last_price"])

                if ticker in tickers and np.isfinite(price) and price > 0:

                    results[ticker] = {
                        "ticker": ticker,
                        "price": price,
                        "timestamp": str(item.get("data_date", "")),
                        "source": "moomoo"
                    }

        except Exception as e:
            print("moomoo unavailable. Trying Yahoo Finance.")

    else:
        print("No moomoo authentication. Trying Yahoo Finance.")

    # 2. Yahoo Finance for missing stocks
    for ticker in tickers:

        if ticker in results:
            continue

        try:
            history = yf.Ticker(ticker).history(period="5d")

            if history.empty:
                raise ValueError("No Yahoo Finance data")

            price = float(history["Close"].iloc[-1])

            if not np.isfinite(price) or price <= 0:
                raise ValueError("Invalid price")

            results[ticker] = {
                "ticker": ticker,
                "price": price,
                "timestamp": str(history.index[-1].date()),
                "source": "yfinance"
            }

        except Exception:
            print(f"{ticker}: Yahoo Finance failed. Using CSV.")

        # 3. CSV fallback
        if ticker not in results:

            row = backup_prices[
                backup_prices["ticker"] == ticker
            ]

            if row.empty:
                raise ValueError(f"No price for {ticker}")

            row = row.iloc[0]

            results[ticker] = {
                "ticker": ticker,
                "price": float(row["price"]),
                "timestamp": str(row["timestamp"]),
                "source": "CSV fallback"
            }

    return pd.DataFrame(results.values())

### 2. Save stock prices into CSV

In [12]:
stock_prices.to_csv("../../data/stock_prices.csv", index=False)

print("Stock prices saved successfully!")
display(stock_prices)

Stock prices saved successfully!


,ticker,price,timestamp,source
0,NVDA,206.870,2026-11-02,moomoo
1,TSLA,410.880,2026-11-02,moomoo
2,AAPL,339.365,2026-11-02,moomoo
3,MSFT,523.845,2026-11-02,moomoo


### 3. Read all CSV files 

In [14]:
accounts = pd.read_csv("../../data/accounts.csv")
positions = pd.read_csv("../../data/positions.csv")
stock_prices = pd.read_csv("../../data/stock_prices.csv")
margin_ratios = pd.read_csv("../../data/margin_ratios.csv")

In [15]:
print("Customer Accounts")
display(accounts.head())

print("Stock Positions")
display(positions.head())

print("Stock Prices")
display(stock_prices.head())

print("Margin Ratios")
display(margin_ratios.head())

Customer Accounts


,account_id,currency,cash,margin_loan,below_mm_since
0,M-001,USD,5000,0,NaN
1,M-002,USD,1000,1500,NaN
2,M-003,USD,0,3000,2026-10-30T10:00:00Z
3,M-004,USD,3000,1000,NaN
4,M-005,USD,500,4000,NaN


Stock Positions


,account_id,ticker,qty,avg_cost
0,M-001,NVDA,30,180
1,M-002,TSLA,10,250
2,M-003,NVDA,20,180
3,M-004,AAPL,10,220
4,M-004,MSFT,5,400


Stock Prices


,ticker,price,timestamp,source
0,NVDA,206.870,2026-11-02,moomoo
1,TSLA,410.880,2026-11-02,moomoo
2,AAPL,339.365,2026-11-02,moomoo
3,MSFT,523.845,2026-11-02,moomoo


Margin Ratios


,ticker,initial_ratio,maintenance_ratio,soft_edge_ratio
0,NVDA,0.4,0.30,0.20
1,TSLA,0.5,0.35,0.25
2,AAPL,0.4,0.30,0.20
3,MSFT,0.4,0.30,0.20


In [16]:
print("Accounts:", accounts.shape)
print("Positions:", positions.shape)
print("Stock Prices:", stock_prices.shape)
print("Margin Ratios:", margin_ratios.shape)

Accounts: (10, 5)
Positions: (12, 4)
Stock Prices: (4, 4)
Margin Ratios: (4, 4)


### 4. Check data quality 

In [18]:
# Check missing values and duplicates
datasets = {
    "Accounts": accounts,
    "Positions": positions,
    "Stock Prices": stock_prices,
    "Margin Ratios": margin_ratios
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Missing values:")
    print(df.isnull().sum())
    print("Duplicate rows:", df.duplicated().sum())


Accounts
Shape: (10, 5)
Missing values:
account_id        0
currency          0
cash              0
margin_loan       0
below_mm_since    8
dtype: int64
Duplicate rows: 0

Positions
Shape: (12, 4)
Missing values:
account_id    0
ticker        0
qty           0
avg_cost      0
dtype: int64
Duplicate rows: 0

Stock Prices
Shape: (4, 4)
Missing values:
ticker       0
price        0
timestamp    0
source       0
dtype: int64
Duplicate rows: 0

Margin Ratios
Shape: (4, 4)
Missing values:
ticker               0
initial_ratio        0
maintenance_ratio    0
soft_edge_ratio      0
dtype: int64
Duplicate rows: 0


In [19]:
# Validate important columns 
# Check unique account IDs and stock records
assert accounts["account_id"].is_unique
assert stock_prices["ticker"].is_unique
assert margin_ratios["ticker"].is_unique

# Check numeric values
assert (accounts["cash"] >= 0).all()
assert (accounts["margin_loan"] >= 0).all()
assert (positions["qty"] > 0).all()
assert (stock_prices["price"] > 0).all()

# Check that account IDs and tickers match
assert positions["account_id"].isin(
    accounts["account_id"]
).all()

assert positions["ticker"].isin(
    stock_prices["ticker"]
).all()

assert positions["ticker"].isin(
    margin_ratios["ticker"]
).all()

print("Data validation passed!")

Data validation passed!


### 5. Merge all four datasets 

In [21]:
portfolio = (
    positions
    .merge(stock_prices, on="ticker", how="left",
           validate="many_to_one")
    .merge(margin_ratios, on="ticker", how="left",
           validate="many_to_one")
    .merge(accounts, on="account_id", how="left",
           validate="many_to_one")
)

display(portfolio.head())

,account_id,ticker,qty,avg_cost,price,timestamp,source,initial_ratio,maintenance_ratio,soft_edge_ratio,currency,cash,margin_loan,below_mm_since
0,M-001,NVDA,30,180,206.870,2026-11-02,moomoo,0.4,0.30,0.20,USD,5000,0,NaN
1,M-002,TSLA,10,250,410.880,2026-11-02,moomoo,0.5,0.35,0.25,USD,1000,1500,NaN
2,M-003,NVDA,20,180,206.870,2026-11-02,moomoo,0.4,0.30,0.20,USD,0,3000,2026-10-30T10:00:00Z
3,M-004,AAPL,10,220,339.365,2026-11-02,moomoo,0.4,0.30,0.20,USD,3000,1000,NaN
4,M-004,MSFT,5,400,523.845,2026-11-02,moomoo,0.4,0.30,0.20,USD,3000,1000,NaN


### 6. Calculation

In [23]:
# market value of each stock position
portfolio["market_value"] = (
    portfolio["price"] * portfolio["qty"]
)

display(
    portfolio[
        ["account_id", "ticker", "qty", "price", "market_value"]
    ]
)

,account_id,ticker,qty,price,market_value
0,M-001,NVDA,30,206.870,6206.100
1,M-002,TSLA,10,410.880,4108.800
2,M-003,NVDA,20,206.870,4137.400
3,M-004,AAPL,10,339.365,3393.650
4,M-004,MSFT,5,523.845,2619.225
5,M-005,TSLA,20,410.880,8217.600
6,M-006,NVDA,20,206.870,4137.400
7,M-006,AAPL,10,339.365,3393.650
8,M-007,MSFT,15,523.845,7857.675
9,M-008,TSLA,10,410.880,4108.800


In [24]:
# total portfolio market value for each customer
account_values = (
    portfolio.groupby("account_id", as_index=False)
    .agg(
        total_market_value=("market_value", "sum")
    )
)

display(account_values)

,account_id,total_market_value
0,M-001,6206.100
1,M-002,4108.800
2,M-003,4137.400
3,M-004,6012.875
4,M-005,8217.600
5,M-006,7531.050
6,M-007,7857.675
7,M-008,4108.800
8,M-009,5090.475
9,M-010,6206.100


In [25]:
# Customer equity 
account_values = account_values.merge(
    accounts[
        ["account_id", "cash", "margin_loan", "below_mm_since"]
    ],
    on="account_id",
    how="left",
    validate="one_to_one"
)

account_values["equity"] = (
    account_values["total_market_value"]
    + account_values["cash"]
    - account_values["margin_loan"]
)

display(account_values)

,account_id,total_market_value,cash,margin_loan,below_mm_since,equity
0,M-001,6206.100,5000,0,NaN,11206.100
1,M-002,4108.800,1000,1500,NaN,3608.800
2,M-003,4137.400,0,3000,2026-10-30T10:00:00Z,1137.400
3,M-004,6012.875,3000,1000,NaN,8012.875
4,M-005,8217.600,500,4000,NaN,4717.600
5,M-006,7531.050,0,6000,NaN,1531.050
6,M-007,7857.675,2000,2000,NaN,7857.675
7,M-008,4108.800,100,4500,2026-11-01T10:00:00Z,-291.200
8,M-009,5090.475,500,2500,NaN,3090.475
9,M-010,6206.100,0,7000,NaN,-793.900


In [26]:
# Calculate margin requirements for each stock position
portfolio["im_req"] = (
    portfolio["market_value"] * portfolio["initial_ratio"]
)

portfolio["mm_req"] = (
    portfolio["market_value"] * portfolio["maintenance_ratio"]
)

portfolio["se_req"] = (
    portfolio["market_value"] * portfolio["soft_edge_ratio"]
)

display(portfolio[
    ["account_id", "ticker", "market_value",
     "im_req", "mm_req", "se_req"]
])

,account_id,ticker,market_value,im_req,mm_req,se_req
0,M-001,NVDA,6206.100,2482.44,1861.8300,1241.220
1,M-002,TSLA,4108.800,2054.40,1438.0800,1027.200
2,M-003,NVDA,4137.400,1654.96,1241.2200,827.480
3,M-004,AAPL,3393.650,1357.46,1018.0950,678.730
4,M-004,MSFT,2619.225,1047.69,785.7675,523.845
5,M-005,TSLA,8217.600,4108.80,2876.1600,2054.400
6,M-006,NVDA,4137.400,1654.96,1241.2200,827.480
7,M-006,AAPL,3393.650,1357.46,1018.0950,678.730
8,M-007,MSFT,7857.675,3143.07,2357.3025,1571.535
9,M-008,TSLA,4108.800,2054.40,1438.0800,1027.200


In [27]:
# Sum margin requirements for each customer
requirements = (
    portfolio.groupby("account_id", as_index=False)
    .agg(
        im_req=("im_req", "sum"),
        mm_req=("mm_req", "sum"),
        se_req=("se_req", "sum")
    )
)

display(requirements)

,account_id,im_req,mm_req,se_req
0,M-001,2482.44,1861.8300,1241.220
1,M-002,2054.40,1438.0800,1027.200
2,M-003,1654.96,1241.2200,827.480
3,M-004,2405.15,1803.8625,1202.575
4,M-005,4108.80,2876.1600,2054.400
5,M-006,3012.42,2259.3150,1506.210
6,M-007,3143.07,2357.3025,1571.535
7,M-008,2054.40,1438.0800,1027.200
8,M-009,2036.19,1527.1425,1018.095
9,M-010,2482.44,1861.8300,1241.220


In [28]:
# Combine account equity with margin requirements
account_values = account_values.merge(
    requirements,
    on="account_id",
    how="left",
    validate="one_to_one"
)

display(account_values)

,account_id,total_market_value,cash,margin_loan,below_mm_since,equity,im_req,mm_req,se_req
0,M-001,6206.100,5000,0,NaN,11206.100,2482.44,1861.8300,1241.220
1,M-002,4108.800,1000,1500,NaN,3608.800,2054.40,1438.0800,1027.200
2,M-003,4137.400,0,3000,2026-10-30T10:00:00Z,1137.400,1654.96,1241.2200,827.480
3,M-004,6012.875,3000,1000,NaN,8012.875,2405.15,1803.8625,1202.575
4,M-005,8217.600,500,4000,NaN,4717.600,4108.80,2876.1600,2054.400
5,M-006,7531.050,0,6000,NaN,1531.050,3012.42,2259.3150,1506.210
6,M-007,7857.675,2000,2000,NaN,7857.675,3143.07,2357.3025,1571.535
7,M-008,4108.800,100,4500,2026-11-01T10:00:00Z,-291.200,2054.40,1438.0800,1027.200
8,M-009,5090.475,500,2500,NaN,3090.475,2036.19,1527.1425,1018.095
9,M-010,6206.100,0,7000,NaN,-793.900,2482.44,1861.8300,1241.220


### 7. Classify Customer Margin Risk 

In [30]:

account_values["status"] = np.select(
    [
        account_values["equity"] < account_values["se_req"],
        account_values["equity"] < account_values["mm_req"]
    ],
    [
        "below_se",
        "below_mm"
    ],
    default="healthy"
)

display(account_values[
    ["account_id", "equity", "mm_req", "se_req", "status"]
])

,account_id,equity,mm_req,se_req,status
0,M-001,11206.100,1861.8300,1241.220,healthy
1,M-002,3608.800,1438.0800,1027.200,healthy
2,M-003,1137.400,1241.2200,827.480,below_mm
3,M-004,8012.875,1803.8625,1202.575,healthy
4,M-005,4717.600,2876.1600,2054.400,healthy
5,M-006,1531.050,2259.3150,1506.210,below_mm
6,M-007,7857.675,2357.3025,1571.535,healthy
7,M-008,-291.200,1438.0800,1027.200,below_se
8,M-009,3090.475,1527.1425,1018.095,healthy
9,M-010,-793.900,1861.8300,1241.220,below_se


### 8. Create get_account() function

In [32]:
def get_account(account_id):

    # Retrieve customer account
    customer = accounts[
        accounts["account_id"] == account_id
    ]

    if customer.empty:
        raise ValueError("Account not found")

    customer = customer.iloc[0]

    # Retrieve customer's stock positions
    customer_positions = positions[
        positions["account_id"] == account_id
    ]

    # Convert stock positions into a list
    position_list = customer_positions[
        ["ticker", "qty", "avg_cost"]
    ].to_dict(orient="records")

    return {
        "account_id": account_id,
        "currency": customer["currency"],
        "cash": float(customer["cash"]),
        "margin_loan": float(customer["margin_loan"]),
        "positions": position_list
    }

In [33]:
# Test the function
get_account("M-004")

{'account_id': 'M-004',
 'currency': 'USD',
 'cash': 3000.0,
 'margin_loan': 1000.0,
 'positions': [{'ticker': 'AAPL', 'qty': 10, 'avg_cost': 220},
  {'ticker': 'MSFT', 'qty': 5, 'avg_cost': 400}]}

### 9. Create get_prices() function

In [35]:
def get_prices(tickers, mode="snapshot"):

    if isinstance(tickers, str):
        tickers = [tickers]

    results = {}

    for ticker in tickers:

        price_row = stock_prices[
            stock_prices["ticker"] == ticker
        ]

        if price_row.empty:
            raise ValueError(f"Price not found: {ticker}")

        price_row = price_row.iloc[0]

        results[ticker] = {
            "price": float(price_row["price"]),
            "timestamp": str(price_row["timestamp"]),
            "source": str(price_row["source"])
        }

    return results

In [36]:
#Test the function
get_prices(["NVDA", "TSLA"])

{'NVDA': {'price': 206.87, 'timestamp': '2026-11-02', 'source': 'moomoo'},
 'TSLA': {'price': 410.88, 'timestamp': '2026-11-02', 'source': 'moomoo'}}

### 10. Create margin_status() function

In [38]:
import pandas as pd

# Convert breach timestamps to datetime
account_values["below_mm_since"] = pd.to_datetime(
    account_values["below_mm_since"],
    utc=True,
    errors="coerce"
)

def margin_status(account_id, now):

    # 1. Retrieve customer's account data
    customer = account_values[
        account_values["account_id"] == account_id
    ]

    if customer.empty:
        raise ValueError("Account not found")

    customer = customer.iloc[0]

    # 2. Convert evaluation time to datetime
    now = pd.Timestamp(now)

    if now.tzinfo is None:
        now = now.tz_localize("UTC")
    else:
        now = now.tz_convert("UTC")

    # 3. Determine margin status
    if customer["equity"] < customer["se_req"]:
        status = "below_se"

    elif customer["equity"] < customer["mm_req"]:
        status = "below_mm"

    else:
        status = "healthy"

    # 4. Calculate hours below maintenance margin
    if status == "healthy":
        hours_below_mm = 0.0

    elif pd.notna(customer["below_mm_since"]):
        hours_below_mm = max(
            0.0,
            (
                now - customer["below_mm_since"]
            ).total_seconds() / 3600
        )

    else:
        hours_below_mm = None

    # 5. Return results
    return {
        "market_value": float(customer["total_market_value"]),
        "equity": float(customer["equity"]),
        "im_req": float(customer["im_req"]),
        "mm_req": float(customer["mm_req"]),
        "se_req": float(customer["se_req"]),
        "status": status,
        "hours_below_mm": hours_below_mm
    }

In [39]:
#Test the function
result = margin_status(
    "M-003",
    "2026-11-02T15:00:00Z"
)

result

{'market_value': 4137.4,
 'equity': 1137.3999999999996,
 'im_req': 1654.96,
 'mm_req': 1241.2199999999998,
 'se_req': 827.48,
 'status': 'below_mm',
 'hours_below_mm': 77.0}

### 11. Create liquidation_distance() function

In [41]:
def liquidation_distance(account_id, shocks=None):

    # Get customer's stock positions
    customer = portfolio[
        portfolio["account_id"] == account_id
    ].copy()

    if customer.empty:
        raise ValueError("Account not found")

    # Apply simulated stock price changes
    shocks = shocks or {}

    customer["stressed_price"] = customer.apply(
        lambda row: row["price"] *
        (1 + shocks.get(row["ticker"], 0)),
        axis=1
    )

    # Calculate stressed market values
    customer["stressed_value"] = (
        customer["qty"] * customer["stressed_price"]
    )

    # Get the customer's account details
    account = accounts[
        accounts["account_id"] == account_id
    ].iloc[0]

    # Calculate stressed equity
    stressed_value = customer["stressed_value"].sum()

    stressed_equity = (
        stressed_value
        + account["cash"]
        - account["margin_loan"]
    )

    # Recalculate stressed margin requirements
    stressed_mm = (
        customer["stressed_value"] *
        customer["maintenance_ratio"]
    ).sum()

    stressed_se = (
        customer["stressed_value"] *
        customer["soft_edge_ratio"]
    ).sum()

    # Calculate distance to each threshold
    # Assuming all positions decline by the same percentage
    current_value = customer["market_value"].sum()
    current_equity = (
        current_value
        + account["cash"]
        - account["margin_loan"]
    )

    current_mm = customer["mm_req"].sum()
    current_se = customer["se_req"].sum()

    mm_buffer = current_equity - current_mm
    se_buffer = current_equity - current_se

    mm_exposure = current_value - current_mm
    se_exposure = current_value - current_se

    pct_drop_mm = (
        max(0, mm_buffer / mm_exposure)
        if mm_exposure > 0 else None
    )

    pct_drop_se = (
        max(0, se_buffer / se_exposure)
        if se_exposure > 0 else None
    )

    # Identify the largest stressed position
    binding_position = customer.loc[
        customer["stressed_value"].idxmax(),
        "ticker"
    ]

    return {
        "pct_drop_to_mm": pct_drop_mm,
        "pct_drop_to_se": pct_drop_se,
        "stressed_equity": float(stressed_equity),
        "binding_position": binding_position
    }

In [42]:
liquidation_distance(
    "M-003",
    shocks={"NVDA": -0.08}
)

{'pct_drop_to_mm': 0,
 'pct_drop_to_se': np.float64(0.09363368298931686),
 'stressed_equity': 806.4080000000004,
 'binding_position': 'NVDA'}

### 12. Create weekend_risk_check() function

In [44]:
def weekend_risk_check(account_id, now):

    # Get customer's current margin status
    risk = margin_status(account_id, now)

    # Convert evaluation time
    now = pd.Timestamp(now)

    # Check whether today is Friday
    is_pre_weekend = now.weekday() == 4

    # Current soft-edge requirement
    current_se = risk["se_req"]

    # Hypothetical 25% increase in soft-edge requirement
    raised_se = current_se * 1.25

    # Determine whether account breaches raised requirement
    would_breach = (
        risk["equity"] < raised_se
        if is_pre_weekend
        else False
    )

    return {
        "is_pre_weekend": is_pre_weekend,
        "se_ratio_now": (
            current_se / risk["market_value"]
            if risk["market_value"] > 0 else None
        ),
        "se_ratio_raised": (
            raised_se / risk["market_value"]
            if risk["market_value"] > 0 else None
        ),
        "would_breach": would_breach
    }

In [45]:
#Test the function
weekend_risk_check(
    "M-003",
    "2026-11-06T15:00:00Z"
)

{'is_pre_weekend': True,
 'se_ratio_now': 0.2,
 'se_ratio_raised': 0.25,
 'would_breach': False}

In [46]:
# test all the 5 functions built
account_id = "M-003"

print("1. Customer Account")
display(get_account(account_id))

print("2. Stock Prices")
display(get_prices(["NVDA"]))

print("3. Margin Status")
display(margin_status(
    account_id,
    "2026-11-06T15:00:00Z"
))

print("4. Liquidation Distance")
display(liquidation_distance(
    account_id,
    shocks={"NVDA": -0.08}
))

print("5. Weekend Risk")
display(weekend_risk_check(
    account_id,
    "2026-11-06T15:00:00Z"
))

1. Customer Account


{'account_id': 'M-003',
 'currency': 'USD',
 'cash': 0.0,
 'margin_loan': 3000.0,
 'positions': [{'ticker': 'NVDA', 'qty': 20, 'avg_cost': 180}]}

2. Stock Prices


{'NVDA': {'price': 206.87, 'timestamp': '2026-11-02', 'source': 'moomoo'}}

3. Margin Status


{'market_value': 4137.4,
 'equity': 1137.3999999999996,
 'im_req': 1654.96,
 'mm_req': 1241.2199999999998,
 'se_req': 827.48,
 'status': 'below_mm',
 'hours_below_mm': 173.0}

4. Liquidation Distance


{'pct_drop_to_mm': 0,
 'pct_drop_to_se': np.float64(0.09363368298931686),
 'stressed_equity': 806.4080000000004,
 'binding_position': 'NVDA'}

5. Weekend Risk


{'is_pre_weekend': True,
 'se_ratio_now': 0.2,
 'se_ratio_raised': 0.25,
 'would_breach': False}

In [47]:
nb = nbformat.read("Data and Margin Engine.ipynb", as_version=4)

code = "\n\n".join(
    cell.source
    for cell in nb.cells
    if cell.cell_type == "code"
    and not cell.source.lstrip().startswith(("!", "%"))
)

tree = ast.parse(code)

# Functions 
function_names = {
    "get_account",
    "get_prices",
    "margin_status",
    "liquidation_distance",
    "weekend_risk_check"
}

functions = [
    ast.get_source_segment(code, node)
    for node in tree.body
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
    and node.name in function_names
]

Path("margin_functions.py").write_text(
    "import pandas as pd\nimport numpy as np\n\n"
    + "\n\n".join(functions)
)

print(f"Exported {len(functions)} functions!")

Exported 5 functions!
